# Phase 8 — PhysioNet 2017 ECG with QPIE encoders + QMBC-Net

Paper 3 companion notebook. Three pipelines benchmarked on the PhysioNet/CinC 2017 AF Challenge:

1. **Phase-1A** — aggregated 492-d features + PathMNIST-MLP (baseline parity with `Phase7_PathMNIST.ipynb`).
2. **Phase-1B** — un-aggregated $(W \times 24)$ + 1D CNN (baseline parity with `Phase7_UWave.ipynb`).
3. **Phase-2** — un-aggregated $(W \times 24)$ + QMBC-Net (cross-basis attention $\to$ temporal self-attention).

Output: `paper_aeon/figures/*.pdf` and tables for `paper_aeon/main.tex`.

Plan: `~/.claude/plans/refactored-strolling-dawn.md`.

## Cell 1 — Imports + device + reproducibility

In [ ]:
import os, sys, math, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sps, stats as spst
from scipy.signal import butter, filtfilt, hilbert
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
import wfdb

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
PROJECT = Path('/Users/eldana/Documents/Quantum/Thesis/QIP')
FIG_DIR = PROJECT / 'paper_aeon' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(s)

set_seed(0)
print('device:', DEVICE)

## Cell 2 — Download PhysioNet 2017 + parse labels + bandpass + length normalisation

Source: <https://physionet.org/content/challenge-2017/1.0.0/>. 8528 single-lead recordings, 300 Hz, 9–60 s. Labels in `REFERENCE.csv` (4 classes: N, A, O, ~).

In [ ]:
# ==================================================================
# Download PhysioNet 2017 + label parsing + bandpass + length normalisation
# Source: https://physionet.org/content/challenge-2017/1.0.0/
# 8528 single-lead ECG, 300 Hz, 9-60 s. REFERENCE.csv holds {N, A, O, ~}.
# ==================================================================

DATA_DIR = PROJECT / 'data' / 'physionet2017'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FS = 300
TARGET_SEC = 30
TARGET_LEN = FS * TARGET_SEC  # 9000
LABEL_MAP = {'N': 0, 'A': 1, 'O': 2, '~': 3}
CLASS_NAMES = ['N', 'A', 'O', 'noise']

def download_physionet2017(dst):
    """Idempotent download of the 2017 challenge training set via wfdb."""
    ref = dst / 'REFERENCE.csv'
    if ref.exists():
        print(f'already present at {dst}')
        return
    print(f'downloading PhysioNet 2017 to {dst} ...')
    wfdb.dl_database('challenge-2017/1.0.0/training', str(dst))
    print('done')

def bandpass(x, lo=0.5, hi=40.0, fs=FS, order=3):
    b, a = butter(order, [lo, hi], btype='band', fs=fs)
    return filtfilt(b, a, x).astype(np.float32)

def fixlen(x, n=TARGET_LEN):
    if len(x) >= n:
        s = (len(x) - n) // 2
        return x[s:s+n]
    return np.pad(x, (0, n - len(x)))

def load_recording(rec_id):
    rec = wfdb.rdrecord(str(DATA_DIR / rec_id))
    x = rec.p_signal[:, 0].astype(np.float32)
    return fixlen(bandpass(x))

def load_all():
    """Load REFERENCE.csv, then all recordings -> (B, 9000) + labels."""
    ref = pd.read_csv(DATA_DIR / 'REFERENCE.csv', header=None, names=['rec', 'label'])
    print('class counts:'); print(ref['label'].value_counts())
    X = np.zeros((len(ref), TARGET_LEN), dtype=np.float32)
    t0 = time.time()
    for i, rec_id in enumerate(ref['rec']):
        X[i] = load_recording(rec_id)
        if (i + 1) % 1000 == 0:
            print(f'  loaded {i+1}/{len(ref)} ({time.time()-t0:.0f}s)')
    y = ref['label'].map(LABEL_MAP).to_numpy()
    return X, y, ref['rec'].to_numpy()

# Run once to populate cache:
# download_physionet2017(DATA_DIR)
# X_raw, y_all, rec_ids = load_all()
# np.savez_compressed(DATA_DIR / 'cached.npz', X=X_raw, y=y_all, rec=rec_ids)
# print('shapes:', X_raw.shape, y_all.shape)


## Cell 3 — Three-channel construction + RobustChannelScaler

Channel 1: bandpassed amplitude. Channel 2: first difference. Channel 3: Hilbert envelope of the 8–16 Hz wavelet detail. RobustChannelScaler ported from `Phase7_UWave.ipynb` (q5/q95 → [0,1]).

In [ ]:
# ==================================================================
# Three-channel construction + RobustChannelScaler
# Channels: raw bandpassed / first difference / Hilbert envelope of cD4 (~9-18 Hz)
# RobustChannelScaler ported from Phase7_UWave.ipynb (q5/q95 -> [0,1])
# ==================================================================

def three_channels(x):
    """x: (N,) bandpassed -> (N, 3)."""
    c1 = x.astype(np.float32)
    c2 = np.diff(c1, prepend=c1[0]).astype(np.float32)
    coeffs = pywt.wavedec(c1, 'db4', level=5)
    rec = [np.zeros_like(c) for c in coeffs]
    rec[4] = coeffs[4]                                 # cD4 ~ 9-18 Hz at fs=300
    band = pywt.waverec(rec, 'db4')[: len(c1)]
    c3 = np.abs(hilbert(band)).astype(np.float32)
    return np.stack([c1, c2, c3], axis=-1)

def make_three_channel_dataset(X_raw):
    """X_raw: (B, N) -> (B, N, 3)."""
    B = len(X_raw)
    out = np.zeros((B, X_raw.shape[1], 3), dtype=np.float32)
    for i in range(B):
        out[i] = three_channels(X_raw[i])
    return out

class RobustChannelScaler:
    def fit(self, X):                                  # X: (B, N, 3)
        flat = X.reshape(-1, X.shape[-1])
        self.q5 = np.quantile(flat, 0.05, axis=0)
        self.q95 = np.quantile(flat, 0.95, axis=0)
        return self
    def transform(self, X):
        Z = (X - self.q5) / (self.q95 - self.q5 + 1e-8)
        return np.clip(Z, 0.0, 1.0).astype(np.float32)


## Cell 4 — Windowing (250 / stride 125)

$W = (9000 - 250) / 125 + 1 = 71$ windows per recording.

In [ ]:
# ==================================================================
# Windowing (250 / stride 125) + train/val/test split
# W = (9000 - 250) / 125 + 1 = 71 windows per recording.
# ==================================================================

WIN = 250
STRIDE = 125

def windowize(X3):
    """X3: (B, N, 3) -> (B, W, WIN, 3)."""
    B, N, C = X3.shape
    W = (N - WIN) // STRIDE + 1
    out = np.zeros((B, W, WIN, C), dtype=np.float32)
    for w in range(W):
        out[:, w] = X3[:, w*STRIDE : w*STRIDE + WIN]
    return out

def stratified_split(y, frac=(0.6, 0.2, 0.2), seed=0):
    """Returns idx_tr, idx_va, idx_te stratified on y."""
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=frac[1] + frac[2], random_state=seed)
    idx_tr, idx_rest = next(sss1.split(np.zeros_like(y), y))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=frac[2] / (frac[1] + frac[2]), random_state=seed)
    idx_va, idx_te = next(sss2.split(np.zeros_like(y[idx_rest]), y[idx_rest]))
    return idx_tr, idx_rest[idx_va], idx_rest[idx_te]

def build_dataset(cache_path=None):
    """End-to-end: load -> three channels -> scale -> window -> split."""
    if cache_path and Path(cache_path).exists():
        d = np.load(cache_path)
        X_raw, y_all = d['X'], d['y']
    else:
        X_raw, y_all, _ = load_all()
    X3 = make_three_channel_dataset(X_raw)
    idx_tr, idx_va, idx_te = stratified_split(y_all, seed=0)
    scaler = RobustChannelScaler().fit(X3[idx_tr])
    X3 = scaler.transform(X3)
    Xw = windowize(X3)                                 # (B, W, WIN, 3)
    return {
        'X_win': Xw, 'y': y_all,
        'idx_tr': idx_tr, 'idx_va': idx_va, 'idx_te': idx_te,
        'scaler': scaler,
    }

# To run end-to-end:
# data = build_dataset(cache_path=DATA_DIR / 'cached.npz')
# print('X_win:', data['X_win'].shape, 'splits:', len(data['idx_tr']), len(data['idx_va']), len(data['idx_te']))


## Cell 5 — Six QPIE encoders + multi-basis projection

Ported verbatim from `Phase7_PathMNIST.ipynb` cell 2 / `Phase7_BloodMNIST.ipynb` cell 3. Encoders are dataset-agnostic — they take a 3-channel sample in $[0,1]^3$ and return a 3-qubit amplitude vector of length 8.

In [ ]:
# ==================================================================
# Six QPIE encoders + multi-basis measurement
# Ported verbatim from Phase7_PathMNIST.ipynb cell 2.
# Input convention: array of shape (M, 3) with values in [0, 255].
# For ECG we feed window samples scaled to [0,1] then * 255.
# ==================================================================

def _theta(rgb):
    return rgb.astype(np.float64) / 255.0 * (np.pi / 2)

def _amps_sep(rgb):
    t = _theta(rgb); c, s = np.cos(t), np.sin(t)
    cR,sR,cG,sG,cB,sB = c[:,0],s[:,0],c[:,1],s[:,1],c[:,2],s[:,2]
    return np.column_stack([cR*cG*cB, sR*cG*cB, cR*sG*cB, sR*sG*cB,
                            cR*cG*sB, sR*cG*sB, cR*sG*sB, sR*sG*sB])

def _amps_crye(rgb):
    t = _theta(rgb); c, s = np.cos(t), np.sin(t)
    cR,sR,cG,sG,cB,sB = c[:,0],s[:,0],c[:,1],s[:,1],c[:,2],s[:,2]
    return np.column_stack([ cR*cG*cB, -sR*sG*cB, -cR*sG*sB, -sR*cG*sB,
                             cR*cG*sB, -sR*sG*sB,  cR*sG*cB,  sR*cG*cB])

def _amps_gbe(rgb):
    t = _theta(rgb); c, s = np.cos(t), np.sin(t)
    cR,sR,cG,sG,cB,sB = c[:,0],s[:,0],c[:,1],s[:,1],c[:,2],s[:,2]
    return np.column_stack([
        cR*cG*cB-sR*sG*sB, sR*cG*cB+cR*sG*sB, cR*sG*cB+sR*cG*sB, sR*sG*cB-cR*cG*sB,
        cR*cG*sB+sR*sG*cB, sR*cG*sB-cR*sG*cB, cR*sG*sB-sR*cG*cB, sR*sG*sB+cR*cG*cB
    ]) / np.sqrt(2)

def _amps_pa_cse(rgb):
    t = _theta(rgb)
    cR,sR = np.cos(t[:,0]), np.sin(t[:,0])
    cG,sG = np.cos(t[:,1]), np.sin(t[:,1])
    cB,sB = np.cos(t[:,2]), np.sin(t[:,2])
    norm = rgb.astype(np.float64) / 255.0
    a_rg = norm[:,0] * norm[:,1] * np.pi / 2
    a_gb = norm[:,1] * norm[:,2] * np.pi / 2
    cGe,sGe = np.cos(t[:,1]+a_rg), np.sin(t[:,1]+a_rg)
    cBe,sBe = np.cos(t[:,2]+a_gb), np.sin(t[:,2]+a_gb)
    return np.column_stack([cR*cG*cB, sR*cGe*cB, cR*sG*cBe, sR*sGe*cBe,
                            cR*cG*sB, sR*cGe*sB, cR*sG*sBe, sR*sGe*sBe])

def apply_ry(state, theta, q):
    c = np.cos(theta/2); s = np.sin(theta/2)
    new = state.copy(); stride = 1 << q
    for a in range(8):
        if a & stride == 0:
            b = a + stride
            new[:, a] = c * state[:, a] - s * state[:, b]
            new[:, b] = s * state[:, a] + c * state[:, b]
    return new

def apply_crz(state, alpha, ctrl, tgt):
    new = state.copy(); cs, ts = 1 << ctrl, 1 << tgt
    p_neg, p_pos = np.exp(-1j * alpha / 2), np.exp(1j * alpha / 2)
    for idx in range(8):
        if idx & cs:
            new[:, idx] = (p_pos if idx & ts else p_neg) * state[:, idx]
    return new

def init_state(N):
    s = np.zeros((N, 8), dtype=complex); s[:, 0] = 1.0; return s

def _amps_cp_2L(rgb):
    t = _theta(rgb); N = len(t); s = init_state(N)
    for layer, perm in enumerate([(0,1,2), (2,0,1)]):
        s = apply_ry(s, t[:,perm[0]], 0)
        s = apply_ry(s, t[:,perm[1]], 1)
        s = apply_ry(s, t[:,perm[2]], 2)
        s = apply_crz(s, t[:,perm[0]]*t[:,perm[1]], 0, 1)
        s = apply_crz(s, t[:,perm[1]]*t[:,perm[2]], 1, 2)
        s = apply_crz(s, t[:,perm[0]]*t[:,perm[2]], 0, 2)
    return s

# Multi-basis measurement (Z + X + Y) -> 24 features per sample
_H = np.array([[1,1],[1,-1]], dtype=complex) / np.sqrt(2)
_HSd = np.array([[1,-1j],[1,1j]], dtype=complex) / np.sqrt(2)
def _kron_n(M, n):
    R = M
    for _ in range(n-1): R = np.kron(R, M)
    return R
_V = {3: [np.eye(8,dtype=complex), _kron_n(_H,3), _kron_n(_HSd,3)]}

def multi_basis(amps, nq=3):
    return np.hstack([np.abs(amps @ V.T)**2 for V in _V[nq]])

# CSE — per-window correlation-scaled entanglement (channels c1/c2/c3 here)
def _amps_cse_window(window_3ch):
    """window_3ch: (n_samples, 3) in [0,1]. Returns (n_samples, 8)."""
    px255 = window_3ch.astype(np.float64) * 255.0
    p = window_3ch.astype(np.float64)
    pc = p - p.mean(0)
    cov = pc.T @ pc / max(len(p) - 1, 1)
    std = np.sqrt(np.diag(cov) + 1e-10)
    corr = cov / (np.outer(std, std) + 1e-10)
    rg = float(np.clip(np.abs(corr[0, 1]), 0.01, 0.99))
    gb = float(np.clip(np.abs(corr[1, 2]), 0.01, 0.99))
    t = _theta(px255)
    cR,sR = np.cos(t[:,0]), np.sin(t[:,0])
    cG,sG = np.cos(t[:,1]), np.sin(t[:,1])
    cB,sB = np.cos(t[:,2]), np.sin(t[:,2])
    a_rg, a_gb = rg * np.pi / 2, gb * np.pi / 2
    cGe,sGe = np.cos(t[:,1]+a_rg), np.sin(t[:,1]+a_rg)
    cBe,sBe = np.cos(t[:,2]+a_gb), np.sin(t[:,2]+a_gb)
    return np.column_stack([cR*cG*cB, sR*cGe*cB, cR*sG*cBe, sR*sGe*cBe,
                            cR*cG*sB, sR*cGe*sB, cR*sG*sBe, sR*sGe*sBe])

ENCODERS = {
    'Sep':    _amps_sep,
    'CRyE':   _amps_crye,
    'GBE':    _amps_gbe,
    'CSE':    'cse',
    'PA-CSE': _amps_pa_cse,
    'CP-2L':  _amps_cp_2L,
}


## Cell 6 — Aggregation (`agg`) — for Phase-1A only

Port `agg(X_flat, npix)` from `Phase7_PathMNIST.ipynb` cell 3 (11 stats: mean/std/skew/kurtosis/5 percentiles + pairwise correlations).

In [ ]:
# ==================================================================
# Aggregation + per-window encode pipeline
# agg(): ported verbatim from Phase7_PathMNIST.ipynb cell 3.
# encode_recordings(): produces both
#   - X_agg : (B, 492) for Phase-1A
#   - X_seq : (B, W, 24) for Phase-1B and Phase-2
# ==================================================================

def agg(X_flat, npix):
    """Statistical aggregation -> 9*d + comb(d,2) features."""
    N = len(X_flat); d = X_flat.shape[1] // npix
    P = X_flat.reshape(N, npix, d)
    mu  = P.mean(1)
    std = P.std(1) + 1e-10
    P_c = P - mu[:, None, :]
    sk = (P_c**3).mean(1) / (std**3)
    ku = (P_c**4).mean(1) / (std**4) - 3.0
    pcts = np.percentile(P, [10, 25, 50, 75, 90], axis=1)
    norms = np.sqrt((P_c**2).sum(1, keepdims=True) + 1e-10)
    P_n = P_c / norms
    C = np.einsum('npi,npj->nij', P_n, P_n) / npix
    ii, jj = np.triu_indices(d, k=1)
    corr = C[:, ii, jj]
    return np.nan_to_num(np.hstack([
        mu, std, sk, ku,
        pcts[0], pcts[1], pcts[2], pcts[3], pcts[4],
        corr
    ]).astype(np.float32))


def encode_window(window_3ch, scheme):
    """
    window_3ch: (n_samples, 3) in [0,1].
    Returns multi-basis features (n_samples, 24) for the given scheme.
    """
    if scheme == 'CSE':
        amps = _amps_cse_window(window_3ch)
    else:
        fn = ENCODERS[scheme]
        amps = fn(window_3ch * 255.0)  # encoders expect [0,255] input
    return multi_basis(amps, nq=3)


def encode_recording(X_win, scheme):
    """
    X_win: (W, WIN, 3) windows of one recording, scaled to [0,1].
    Returns:
        seq : (W, 24)   per-window mean over WIN samples (for CNN/QMBC-Net).
        feat: (W*24,)   raw mean-features flattened (used by agg() with npix=W).
    """
    W = X_win.shape[0]
    seq = np.zeros((W, 24), dtype=np.float32)
    for w in range(W):
        mb = encode_window(X_win[w], scheme)   # (WIN, 24)
        seq[w] = mb.mean(axis=0)
    return seq


def encode_dataset(X_win_all, scheme, batch_log=200):
    """
    X_win_all: (B, W, WIN, 3) all recordings windowed, scaled to [0,1].
    Returns:
        X_seq: (B, W, 24)
        X_agg: (B, 492)   via agg() over windows with npix=W
    """
    B, W, _, _ = X_win_all.shape
    X_seq = np.zeros((B, W, 24), dtype=np.float32)
    t0 = time.time()
    for i in range(B):
        X_seq[i] = encode_recording(X_win_all[i], scheme)
        if batch_log and (i + 1) % batch_log == 0:
            print(f'  [{scheme}] {i+1}/{B} ({time.time()-t0:.0f}s)')
    X_flat = X_seq.reshape(B, W * 24)
    X_agg = agg(X_flat, npix=W)
    return X_seq, X_agg


## Cell 7 — Phase-1A: PathMNIST-MLP, 5 seeds

$d \to 512 \to 256 \to 128 \to 4$, BN/ReLU/Dropout 0.3. AdamW 5e-4, cosine, label smoothing 0.05, class-balanced sampler. Macro-F1 over N/A/O. Paired t-test best entangling vs Sep.

In [ ]:
# ==================================================================
# Phase-1A — PathMNIST-MLP on aggregated 492-d features
# 5 seeds, AdamW lr 5e-4 wd 1e-3, cosine, label smoothing 0.05,
# class-balanced sampler, batch 64, 50 epochs, early stopping pat=10.
# Macro-F1 over N/A/O is the official metric.
# ==================================================================

N_SEEDS = 5
N_CLASSES = 4
NAO = [0, 1, 2]                                # N, A, O — Noise excluded from macro-F1

class PathMLP(nn.Module):
    def __init__(self, d_in, n_classes=N_CLASSES, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.net(x)

def class_balanced_sampler(y):
    counts = np.bincount(y, minlength=N_CLASSES).astype(np.float64)
    w = 1.0 / np.maximum(counts, 1)
    sample_w = w[y]
    return WeightedRandomSampler(sample_w, num_samples=len(y), replacement=True)

def macro_f1_nao(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=NAO, average='macro', zero_division=0)

def auc_macro_ovr(y_true, probs):
    try:
        return roc_auc_score(y_true, probs, multi_class='ovr', average='macro')
    except ValueError:
        return float('nan')

def train_one(model, Xtr, ytr, Xva, yva, Xte, yte, seed,
              epochs=50, batch_size=64, lr=5e-4, wd=1e-3, ls=0.05, patience=10):
    set_seed(seed)
    Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr).long()
    Xva_t = torch.from_numpy(Xva).to(DEVICE); yva_t = torch.from_numpy(yva).long()
    Xte_t = torch.from_numpy(Xte).to(DEVICE)
    sampler = class_balanced_sampler(ytr)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, sampler=sampler)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=ls)
    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss_fn(model(xb), yb).backward()
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            pred_va = model(Xva_t).argmax(1).cpu().numpy()
        f1 = macro_f1_nao(yva, pred_va)
        if f1 > best_f1:
            best_f1, best_state, bad = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(Xte_t)
        probs = torch.softmax(logits, 1).cpu().numpy()
        pred = logits.argmax(1).cpu().numpy()
    if hasattr(torch.mps, 'empty_cache'): torch.mps.empty_cache()
    return {
        'macro_f1': macro_f1_nao(yte, pred),
        'auc':      auc_macro_ovr(yte, probs),
        'bacc':     balanced_accuracy_score(yte, pred),
        'pred':     pred, 'probs': probs,
    }

def run_phase1a(data, schemes=None, seeds=N_SEEDS):
    """Runs MLP-on-agg over each scheme x seed. Returns dict."""
    schemes = schemes or list(ENCODERS.keys())
    results = {}
    for scheme in schemes:
        print(f'\n=== Phase-1A {scheme} ===')
        X_seq, X_agg = encode_dataset(data['X_win'], scheme)
        Xtr = X_agg[data['idx_tr']]; ytr = data['y'][data['idx_tr']]
        Xva = X_agg[data['idx_va']]; yva = data['y'][data['idx_va']]
        Xte = X_agg[data['idx_te']]; yte = data['y'][data['idx_te']]
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
        Xtr = (Xtr - mu) / sd; Xva = (Xva - mu) / sd; Xte = (Xte - mu) / sd
        seed_runs = []
        for s in range(seeds):
            model = PathMLP(d_in=Xtr.shape[1])
            r = train_one(model, Xtr, ytr, Xva, yva, Xte, yte, seed=s)
            seed_runs.append(r)
            print(f'  seed {s}: macro-F1={r["macro_f1"]:.4f}  AUC={r["auc"]:.4f}  BAcc={r["bacc"]:.4f}')
        results[scheme] = seed_runs
    return results

# results_1a = run_phase1a(data)


## Cell 8 — Phase-1B: 1D CNN over windows, 5 seeds

`CNN1D` ported from `Phase7_UWave.ipynb`. Conv1d dilations {1,2,4} → AdaptiveAvgPool1d → Linear. Same training recipe.

In [ ]:
# ==================================================================
# Phase-1B — 1D CNN over windows
# Input: (B, W, 24) per-window multi-basis features.
# Conv1d dilations {1,2,4} -> AdaptiveAvgPool1d -> Linear.
# Same training recipe as Phase-1A.
# ==================================================================

class CNN1D(nn.Module):
    def __init__(self, c_in=24, n_classes=N_CLASSES, ch=64, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, ch, 3, padding=1, dilation=1), nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(ch, ch, 3, padding=2, dilation=2),   nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(ch, ch, 3, padding=4, dilation=4),   nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(p),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(ch, n_classes),
        )
    def forward(self, x):                              # x: (B, W, 24) -> (B, 24, W)
        return self.net(x.transpose(1, 2))

def standardise_seq(Xtr, Xva, Xte):
    mu = Xtr.mean(axis=(0, 1), keepdims=True)
    sd = Xtr.std(axis=(0, 1), keepdims=True) + 1e-8
    return (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

def run_phase1b(data, schemes=None, seeds=N_SEEDS):
    schemes = schemes or list(ENCODERS.keys())
    results = {}
    for scheme in schemes:
        print(f'\n=== Phase-1B {scheme} ===')
        X_seq, _ = encode_dataset(data['X_win'], scheme)
        Xtr = X_seq[data['idx_tr']]; ytr = data['y'][data['idx_tr']]
        Xva = X_seq[data['idx_va']]; yva = data['y'][data['idx_va']]
        Xte = X_seq[data['idx_te']]; yte = data['y'][data['idx_te']]
        Xtr, Xva, Xte = standardise_seq(Xtr, Xva, Xte)
        seed_runs = []
        for s in range(seeds):
            model = CNN1D(c_in=24)
            r = train_one(model, Xtr, ytr, Xva, yva, Xte, yte, seed=s)
            seed_runs.append(r)
            print(f'  seed {s}: macro-F1={r["macro_f1"]:.4f}  AUC={r["auc"]:.4f}  BAcc={r["bacc"]:.4f}')
        results[scheme] = seed_runs
    return results

# results_1b = run_phase1b(data)


## Cell 9 — Phase-2: QMBC-Net, 5 seeds

Cross-basis attention over (Z, X, Y) tokens per window → basis gate → temporal self-attention over $W$ windows → CLS → Linear. Returns logits + attention weights for inspection.

In [ ]:
# ==================================================================
# Phase-2 — QMBC-Net (Quantum Multi-Basis Cross-attention Network)
# Per window: 24 features split into 3 basis tokens of 8 (Z, X, Y).
# Cross-basis MHA(4 heads) -> per-basis gate -> pooled window vector.
# Sinusoidal pos enc over windows + 2-layer temporal self-attention(4 heads).
# CLS token -> Linear(N_CLASSES). Returns logits + attention for inspection.
# ==================================================================

def sinusoidal_pos(n, d):
    pos = np.arange(n)[:, None]
    i = np.arange(d)[None]
    angle = pos / (10000 ** (2 * (i // 2) / d))
    pe = np.zeros((n, d), dtype=np.float32)
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return torch.from_numpy(pe)

class QMBCNet(nn.Module):
    def __init__(self, n_classes=N_CLASSES, d=64, n_heads=4, n_temporal_layers=2,
                 max_W=80, dropout=0.1):
        super().__init__()
        self.d = d
        self.basis_embed = nn.Linear(8, d)
        self.basis_attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.basis_gate = nn.Linear(d, 1)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.trunc_normal_(self.cls, std=0.02)
        self.register_buffer('pos', sinusoidal_pos(max_W + 1, d))
        enc_layer = nn.TransformerEncoderLayer(d, n_heads, dim_feedforward=4*d,
                                               dropout=dropout, batch_first=True,
                                               activation='gelu')
        self.temporal = nn.TransformerEncoder(enc_layer, n_temporal_layers)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)

    def forward(self, x, return_attn=False):           # x: (B, W, 24)
        B, W, _ = x.shape
        bases = x.view(B, W, 3, 8)                     # split into Z, X, Y groups
        h = self.basis_embed(bases)                    # (B, W, 3, d)
        h = h.view(B*W, 3, self.d)
        h_attn, w_basis = self.basis_attn(h, h, h, need_weights=True, average_attn_weights=False)
        gate = torch.softmax(self.basis_gate(h_attn).squeeze(-1), dim=-1)   # (B*W, 3)
        win = (h_attn * gate.unsqueeze(-1)).sum(dim=1)                      # (B*W, d)
        win = win.view(B, W, self.d)
        cls = self.cls.expand(B, -1, -1)
        seq = torch.cat([cls, win], dim=1) + self.pos[: W + 1].unsqueeze(0)
        out = self.temporal(seq)
        logits = self.head(self.norm(out[:, 0]))
        if return_attn:
            return logits, {
                'basis': w_basis.view(B, W, -1, 3, 3),                      # per-window cross-basis
                'gate':  gate.view(B, W, 3),                                # basis gate weights
            }
        return logits

def run_phase2(data, schemes=None, seeds=N_SEEDS, epochs=50):
    schemes = schemes or list(ENCODERS.keys())
    results = {}
    for scheme in schemes:
        print(f'\n=== Phase-2 QMBC-Net {scheme} ===')
        X_seq, _ = encode_dataset(data['X_win'], scheme)
        Xtr = X_seq[data['idx_tr']]; ytr = data['y'][data['idx_tr']]
        Xva = X_seq[data['idx_va']]; yva = data['y'][data['idx_va']]
        Xte = X_seq[data['idx_te']]; yte = data['y'][data['idx_te']]
        Xtr, Xva, Xte = standardise_seq(Xtr, Xva, Xte)
        seed_runs = []
        for s in range(seeds):
            model = QMBCNet(max_W=Xtr.shape[1])
            r = train_one(model, Xtr, ytr, Xva, yva, Xte, yte, seed=s, epochs=epochs)
            seed_runs.append(r)
            print(f'  seed {s}: macro-F1={r["macro_f1"]:.4f}  AUC={r["auc"]:.4f}  BAcc={r["bacc"]:.4f}')
        results[scheme] = seed_runs
    return results

# results_2 = run_phase2(data)


## Cell 10 — Combined results table + comparison figures

Saves to `paper_aeon/figures/`:
- `fig_three_channels.pdf` — raw / first-diff / envelope on a sample beat.
- `fig_phase1_results.pdf` — Phase-1A vs Phase-1B macro-F1 bars.
- `fig_qmbc_arch.pdf` — schematic (drawn separately or via TikZ).
- `fig_sota_compare.pdf` — Clifford 2017 / Andreotti / Goodfellow / Hannun + Phase-1A/1B/2.

In [ ]:
# ==================================================================
# Combined results table + comparison figures
# Saves PDFs to paper_aeon/figures/.
# ==================================================================

def summarise(results):
    """results: {scheme: [{macro_f1, auc, bacc, ...}, ...]} -> DataFrame."""
    rows = []
    for scheme, runs in results.items():
        f1s   = [r['macro_f1'] for r in runs]
        aucs  = [r['auc']      for r in runs]
        baccs = [r['bacc']     for r in runs]
        rows.append({
            'scheme': scheme,
            'macro_f1_mean': np.mean(f1s),  'macro_f1_std': np.std(f1s),
            'auc_mean':      np.mean(aucs), 'auc_std':      np.std(aucs),
            'bacc_mean':     np.mean(baccs),'bacc_std':     np.std(baccs),
        })
    return pd.DataFrame(rows)

def paired_t_vs_sep(results, ref='Sep'):
    rows = []
    base = [r['macro_f1'] for r in results[ref]]
    for scheme, runs in results.items():
        if scheme == ref: continue
        cur = [r['macro_f1'] for r in runs]
        diff = np.mean(cur) - np.mean(base)
        t_stat, p = spst.ttest_rel(cur, base)
        rows.append({'scheme': scheme, 'd_macro_f1': diff, 't': t_stat, 'p': p})
    return pd.DataFrame(rows)

def to_latex_table(df, caption, label, cols=('macro_f1', 'auc', 'bacc')):
    lines = [
        r'\begin{table}[t]',
        rf'\caption{{{caption}}}',
        rf'\label{{{label}}}',
        r'\centering',
        r'\begin{tabular}{l' + 'c' * len(cols) + '}',
        r'\toprule',
        'Scheme & ' + ' & '.join(c.replace('_', '-') for c in cols) + r' \\',
        r'\midrule',
    ]
    for _, r in df.iterrows():
        cells = [r['scheme']]
        for c in cols:
            cells.append(f"{r[c+'_mean']:.3f} $\\pm$ {r[c+'_std']:.3f}")
        lines.append(' & '.join(cells) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

def fig_phase1_results(df_1a, df_1b, save=True):
    schemes = df_1a['scheme'].tolist()
    x = np.arange(len(schemes))
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(x - 0.2, df_1a['macro_f1_mean'], 0.4, yerr=df_1a['macro_f1_std'],
           label='Phase-1A (MLP-agg)', capsize=3)
    ax.bar(x + 0.2, df_1b['macro_f1_mean'], 0.4, yerr=df_1b['macro_f1_std'],
           label='Phase-1B (1D CNN)', capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(schemes, rotation=20)
    ax.set_ylabel('macro-F1 (N/A/O)'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save:
        for ext in ('pdf', 'png'): plt.savefig(FIG_DIR / f'fig_phase1_results.{ext}')
    plt.show()

def fig_three_channels(X_raw_one, save=True):
    ch = three_channels(X_raw_one)
    t = np.arange(len(X_raw_one)) / FS
    seg = slice(0, 3 * FS)                             # first 3 s
    fig, axes = plt.subplots(3, 1, figsize=(7, 4), sharex=True)
    titles = ['c$_1$ raw bandpassed', 'c$_2$ first difference', 'c$_3$ Hilbert envelope of cD4']
    for k in range(3):
        axes[k].plot(t[seg], ch[seg, k], lw=0.8, color='k')
        axes[k].set_ylabel(titles[k], fontsize=8)
        axes[k].grid(alpha=0.3)
    axes[-1].set_xlabel('time (s)')
    plt.tight_layout()
    if save:
        for ext in ('pdf', 'png'): plt.savefig(FIG_DIR / f'fig_three_channels.{ext}')
    plt.show()

def fig_sota_compare(df_1a, df_1b, df_2, save=True):
    sota = [
        ('Clifford 2017',  0.83),
        ('Andreotti 2017', 0.82),
        ('Goodfellow 2018',0.83),
        ('Hannun 2019$^\\dagger$', 0.83),
    ]
    ours = [
        ('Phase-1A (best)', df_1a['macro_f1_mean'].max()),
        ('Phase-1B (best)', df_1b['macro_f1_mean'].max()),
        ('Phase-2 (best)',  df_2['macro_f1_mean'].max()),
    ]
    rows = sota + ours
    names = [r[0] for r in rows]; vals = [r[1] for r in rows]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    bars = ax.barh(np.arange(len(rows)), vals,
                   color=['0.7']*len(sota) + ['#1f77b4']*len(ours))
    ax.set_yticks(np.arange(len(rows))); ax.set_yticklabels(names)
    ax.set_xlim(0.6, 1.0); ax.set_xlabel('macro-F1 (N/A/O)')
    ax.axvline(0.83, ls='--', color='red', alpha=0.5, label='Clifford 2017 baseline')
    ax.legend(loc='lower right'); ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    if save:
        for ext in ('pdf', 'png'): plt.savefig(FIG_DIR / f'fig_sota_compare.{ext}')
    plt.show()

# Example invocation once results are produced:
# df_1a = summarise(results_1a); print(df_1a)
# df_1b = summarise(results_1b); print(df_1b)
# df_2  = summarise(results_2);  print(df_2)
# print(to_latex_table(df_1a, 'Phase-1A — aggregated MLP, PhysioNet 2017, 5 seeds.', 'tab:phase1a'))
# print(to_latex_table(df_1b, 'Phase-1B — 1D CNN over windows, PhysioNet 2017, 5 seeds.', 'tab:phase1b'))
# print(to_latex_table(df_2,  'Phase-2 — QMBC-Net, PhysioNet 2017, 5 seeds.', 'tab:phase2'))
# fig_phase1_results(df_1a, df_1b)
# fig_three_channels(X_raw[0])
# fig_sota_compare(df_1a, df_1b, df_2)


## Cell 11 — Attention overlay on AF + Normal example

Pick one held-out AF and one Normal recording. Run QMBC-Net with `return_attn=True`. Overlay temporal attention on the raw ECG; report basis-gate weights in the figure caption.

In [ ]:
# ==================================================================
# Attention overlay on AF + Normal example
# Re-instantiates QMBC-Net on the best scheme and runs a sample of held-out
# AF and Normal recordings to extract temporal + basis-gate attention.
# ==================================================================

def get_attention(model, x_seq):
    """x_seq: (B, W, 24). Returns (logits, attn dict)."""
    model.eval()
    with torch.no_grad():
        x = torch.from_numpy(x_seq).to(DEVICE)
        return model(x, return_attn=True)

def fig_attention(model, X_seq_te, y_te, X_raw_te, save=True):
    """One AF + one Normal recording, ECG with temporal attention overlay."""
    idx_n = np.where(y_te == 0)[0][0]
    idx_a = np.where(y_te == 1)[0][0]
    picks = [('Normal', idx_n), ('AF', idx_a)]
    fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.5), sharex=False)
    for ax, (label, i) in zip(axes, picks):
        x = X_seq_te[i:i+1]
        _, attn = get_attention(model, x)
        gate = attn['gate'][0].cpu().numpy()           # (W, 3)
        # Temporal attention from the [CLS] row, layer-0, mean over heads:
        # NB: nn.TransformerEncoder doesn't expose per-head weights by default;
        # this overlay uses the basis-gate magnitude as a proxy if needed.
        proxy = gate.mean(axis=1)                       # (W,) — replace if you hook the encoder
        t = np.arange(len(X_raw_te[i])) / FS
        ax.plot(t, X_raw_te[i], lw=0.6, color='k')
        # Map per-window attention back to time:
        for w in range(len(proxy)):
            t0 = w * STRIDE / FS; t1 = (w * STRIDE + WIN) / FS
            ax.axvspan(t0, t1, alpha=0.15 * proxy[w] / (proxy.max() + 1e-8),
                       color='red', linewidth=0)
        ax.set_title(f'{label} (gate Z/X/Y mean = '
                     f'{gate[:, 0].mean():.2f} / {gate[:, 1].mean():.2f} / {gate[:, 2].mean():.2f})')
        ax.set_xlabel('time (s)'); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save:
        for ext in ('pdf', 'png'): plt.savefig(FIG_DIR / f'fig_attention.{ext}')
    plt.show()

# After Phase-2 run, retrain the best scheme once with seed 0 and call:
# best_scheme = df_2.sort_values('macro_f1_mean', ascending=False).iloc[0]['scheme']
# X_seq, _ = encode_dataset(data['X_win'], best_scheme)
# Xtr_n, Xva_n, Xte_n = standardise_seq(X_seq[data['idx_tr']], X_seq[data['idx_va']], X_seq[data['idx_te']])
# model = QMBCNet(max_W=Xte_n.shape[1])
# train_one(model, Xtr_n, data['y'][data['idx_tr']],
#           Xva_n, data['y'][data['idx_va']],
#           Xte_n, data['y'][data['idx_te']], seed=0)
# fig_attention(model, Xte_n, data['y'][data['idx_te']], X_raw[data['idx_te']])
